# Tome 11, Chapter 8: Random walks on networks and groups (Python companion)

Exercises first, solutions at the end. Effective resistances computed by linear algebra and checked against walks, Pólya's constant in $\mathbb Z^3$ obtained from Ohm's law, the logarithm of $\mathbb Z^2$, Kesten's exponential decay on the free group, PageRank as a random walk, and a computer science application: **spectral sparsification by effective resistances** (Spielman and Srivastava, 2008), which keeps a few edges of a dense graph, chosen by their resistance, and preserves every cut and every Laplacian quadratic form up to a small factor. Requires `numpy` and `scipy`.

In [1]:
import math
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spl
rng = np.random.default_rng(1108)

def laplacian(n, edges, w=None):
    # dense Laplacian of a weighted graph on {0,...,n-1}
    L = np.zeros((n, n)); w = np.ones(len(edges)) if w is None else w
    for (a, b), c in zip(edges, w):
        L[a, a] += c; L[b, b] += c; L[a, b] -= c; L[b, a] -= c
    return L

def hypercube(n):
    return [(x, x ^ (1 << i)) for x in range(2**n) for i in range(n) if x < x ^ (1 << i)]

## Exercises

**P8.1 (★) Resistance by linear algebra.** Show that $R(a\leftrightarrow b)=(e_a-e_b)^{\mathsf T}L^+(e_a-e_b)$, where $L^+$ is the pseudo-inverse of the Laplacian. Compute $R(0\leftrightarrow\mathbf 1)$ on the hypercube $\{0,1\}^n$ for $n=2,\dots,10$ and compare with the formula of Exercise 8.3. For $n=6$, estimate the commute time between opposite corners by simulation and compare with Theorem 8.5.

**P8.2 (★★) Pólya's constant from Ohm's law.** In $\mathbb Z^3$, short the boundary of the box $[-n,n]^3$ and compute $R_n=R(0\leftrightarrow\text{boundary})$ by solving the sparse linear system, for $n=5,10,20,30$. Deduce the probability of return to $0$ via Theorem 8.6, extrapolate to $n=\infty$ and compare with the known value $0.3405\ldots$ (Watson, 1939).

**P8.3 (★) The logarithm of the plane.** Do the same in $\mathbb Z^2$ for $n=8,16,\dots,128$. Show numerically that $R_n$ grows by $\frac{\log2}{2\pi}$ each time $n$ doubles.

**P8.4 (★★) Kesten's decay.** Compute exactly $p_{2n}=P(X_{2n}=e)$ for simple random walk on the free group $F_2$, $n\le200$, by dynamic programming on the distance chain. Check that $p_{2n}^{1/(2n)}\to\frac{\sqrt3}2$, and estimate the polynomial correction $p_{2n}/(\sqrt3/2)^{2n}\approx Cn^{-\beta}$.

**P8.5 (★★) PageRank is a random walk.** On a directed graph, the PageRank of Brin and Page (1998) is the stationary law of the walk that follows a uniform outgoing link with probability $0.85$ and jumps to a uniform page otherwise. Compute it on a random directed graph with $2000$ pages by power iteration, then estimate it by running short walks ending at the first teleportation (the Monte Carlo method of Avrachenkov et al., 2007), and compare.

**P8.6 (★★★) CS application: sparsification by effective resistances.** Take two dense random graphs on $150$ vertices each (about $11\,000$ edges each), joined by a single edge, a caricature of two communities in a social network. Sample $q$ edges with replacement, edge $e$ with probability $p_e\propto R_e$ (its effective resistance), each sample with weight $\frac1{qp_e}$. Compare the Laplacian quadratic forms $x^{\mathsf T}Lx$ and $x^{\mathsf T}\tilde Lx$ for random $x$ and for cut vectors, and the extreme eigenvalues of $L^{+/2}\tilde LL^{+/2}$, for several $q$; compare with uniform sampling of edges.

## Solutions

### P8.1

**How to find it.** The potential $u$ of the unit current solves $Lu=e_a-e_b$, and $e_a-e_b$ is orthogonal to the constants, the kernel of $L$, so $u=L^+(e_a-e_b)$ up to a constant, and $R=u(a)-u(b)$. For the walk, simulate until $\mathbf 1$ is hit, then until $0$ is hit again.

In [2]:
for n in range(2, 11):
    L = laplacian(2**n, hypercube(n)); Lp = np.linalg.pinv(L)
    e = np.zeros(2**n); e[0] = 1; e[-1] = -1
    formula = sum(1 / ((n - j) * math.comb(n, j)) for j in range(n))
    print(f"n = {n:>2}: R = {e @ Lp @ e:.6f}   formula {formula:.6f}   n R = {n * formula:.4f}")

n = 6; N = 2**n; target = N - 1
def hit(start, goal):
    x, t = start, 0
    while x != goal:
        x ^= 1 << rng.integers(n); t += 1
    return t
times = [hit(0, target) + hit(target, 0) for _ in range(4000)]
R6 = sum(1 / ((n - j) * math.comb(n, j)) for j in range(n))
print(f"commute time, n = 6: simulated {np.mean(times):.1f} +- {np.std(times) / math.sqrt(4000):.1f},  c(G) R = {n * 2**n * R6:.1f}")

n =  2: R = 1.000000   formula 1.000000   n R = 2.0000
n =  3: R = 0.833333   formula 0.833333   n R = 2.5000
n =  4: R = 0.666667   formula 0.666667   n R = 2.6667
n =  5: R = 0.533333   formula 0.533333   n R = 2.6667
n =  6: R = 0.433333   formula 0.433333   n R = 2.6000


n =  7: R = 0.359524   formula 0.359524   n R = 2.5167
n =  8: R = 0.304764   formula 0.304762   n R = 2.4381


n =  9: R = 0.263492   formula 0.263492   n R = 2.3714


n = 10: R = 0.231747   formula 0.231746   n R = 2.3175


commute time, n = 6: simulated 168.3 +- 1.7,  c(G) R = 166.4


The linear algebra reproduces the shorting formula to machine precision, and $nR\to2$ slowly (the correction is of order $\frac1n$). The simulated commute time agrees with $c(G)R=n2^nR$, within its standard error; the walk needs about $2^{n+1}$ steps to go to the opposite corner and back, almost all of them spent near the middle levels, where most of the cube is.

### P8.2

**How to find it.** Number the vertices of the box, keep the interior ones as unknowns, set the boundary to $0$ volts, and solve $L_Iu=e_0$ by conjugate gradients ($L_I$, the Laplacian restricted to the interior, is symmetric positive definite). Then $R_n=u(0)$ and the return probability is $1-\frac1{6R_n}$ (Theorem 8.6 with $\pi(0)=6$). The finite-size error is of order $\frac1n$, so extrapolate linearly in $\frac1n$.

In [3]:
def box_resistance(d, n):
    side = 2 * n + 1; N = side**d
    idx = np.arange(N).reshape((side,) * d)
    rows, cols = [], []
    for ax in range(d):
        a = np.take(idx, range(side - 1), axis=ax).ravel(); b = np.take(idx, range(1, side), axis=ax).ravel()
        rows += [a, b]; cols += [b, a]
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    A = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N))
    L = sp.diags(np.full(N, 2.0 * d)) - A
    coords = np.array(np.unravel_index(np.arange(N), (side,) * d)).T - n
    I = np.where(np.all(np.abs(coords) < n, axis=1))[0]
    LI = L[I][:, I].tocsc()
    o = np.where(np.all(coords[I] == 0, axis=1))[0][0]
    e = np.zeros(len(I)); e[o] = 1
    u, info = spl.cg(LI, e, rtol=1e-10, maxiter=20_000)
    return u[o]

R3 = {}
for n in (5, 10, 20, 30):
    R3[n] = box_resistance(3, n)
    print(f"n = {n:>2}: R_n = {R3[n]:.5f}, return probability {1 - 1 / (6 * R3[n]):.4f}")
Rinf = (30 * R3[30] - 20 * R3[20]) / 10
print(f"extrapolated: R = {Rinf:.5f}, return probability {1 - 1 / (6 * Rinf):.4f}  (Watson: 0.3405)")

n =  5: R_n = 0.23870, return probability 0.3018
n = 10: R_n = 0.24576, return probability 0.3218


n = 20: R_n = 0.24925, return probability 0.3313


n = 30: R_n = 0.25041, return probability 0.3344
extrapolated: R = 0.25273, return probability 0.3405  (Watson: 0.3405)


The finite boxes underestimate the resistance (the shorted boundary is a shortcut), and the return probability increases with $n$; the linear extrapolation in $\frac1n$ from $n=20$ and $30$ gives $0.340$, the classical value $0.3405$ computed by G. N. Watson in 1939 from an integral of elliptic functions (*Three triple integrals*, Quarterly Journal of Mathematics 10, 1939). Ohm's law and a linear solver replace the elliptic integrals.

### P8.3

**How to find it.** The same function with $d=2$. A conducting disc of radius $n$ in the continuous plane has resistance $\frac1{2\pi}\log n+$const to its centre, and the lattice behaves the same way.

In [4]:
prev = None
for n in (8, 16, 32, 64, 128):
    R = box_resistance(2, n)
    print(f"n = {n:>3}: R_n = {R:.5f}" + (f"   increase {R - prev:.5f}" if prev else ""))
    prev = R
print(f"log 2 / (2 pi) = {math.log(2) / (2 * math.pi):.5f}")

n =   8: R_n = 0.60009
n =  16: R_n = 0.71061   increase 0.11051
n =  32: R_n = 0.82097   increase 0.11037


n =  64: R_n = 0.93130   increase 0.11033


n = 128: R_n = 1.04162   increase 0.11032
log 2 / (2 pi) = 0.11032


Each doubling adds between $0.1103$ and $0.1105$ to the resistance, matching $\frac{\log2}{2\pi}=0.1103$: $R_n\approx\frac1{2\pi}\log n+c$, unbounded, so the walk is recurrent (Theorem 8.6), but only logarithmically. The escape probability $\frac1{4R_n}$ from $0$ to distance $n$ decays like $\frac{\pi}{2\log n}$: to escape to distance $10^6$ before returning has probability only about $0.10$.

### P8.4

**How to find it.** The distance $|X_n|$ is a Markov chain on $\{0,1,\dots\}$: from $0$ to $1$; from $j\ge1$ up with probability $\frac34$, down with probability $\frac14$ (for $k=2$). Propagate its law exactly and read $p_{2n}=P(|X_{2n}|=0)$.

In [5]:
k = 2; p, q = (2 * k - 1) / (2 * k), 1 / (2 * k)
rho = math.sqrt(2 * k - 1) / k
dist = np.zeros(402); dist[0] = 1.0
ret = {}
for t in range(1, 401):
    new = np.zeros_like(dist)
    new[1] += dist[0]
    new[2:] += p * dist[1:-1]
    new[0:-2] += q * dist[1:-1]
    dist = new
    if t % 2 == 0:
        ret[t // 2] = dist[0]
for n in (10, 50, 100, 200):
    print(f"n = {n:>3}: p_2n = {ret[n]:.3e}, p_2n^(1/2n) = {ret[n]**(1 / (2 * n)):.4f}, p_2n / rho^2n = {ret[n] / rho**(2 * n):.5f}")
beta = -math.log((ret[200] / rho**400) / (ret[100] / rho**200)) / math.log(2)
print(f"rho = {rho:.4f};  estimated polynomial exponent beta = {beta:.3f}")

n =  10: p_2n = 2.038e-03, p_2n^(1/2n) = 0.7336, p_2n / rho^2n = 0.03619
n =  50: p_2n = 2.449e-09, p_2n^(1/2n) = 0.8201, p_2n / rho^2n = 0.00432
n = 100: p_2n = 5.147e-16, p_2n^(1/2n) = 0.8386, p_2n / rho^2n = 0.00160
n = 200: p_2n = 5.989e-29, p_2n^(1/2n) = 0.8500, p_2n / rho^2n = 0.00058
rho = 0.8660;  estimated polynomial exponent beta = 1.463


The $(2n)$-th roots approach $\frac{\sqrt3}2=0.8660$ slowly, because of a polynomial factor: $p_{2n}\approx C\rho^{2n}n^{-3/2}$, the exponent $\frac32$ being that of the Catalan numbers in the proof of Theorem 8.13 (paths returning for the first time). On $\mathbb Z^d$ the decay is polynomial, $n^{-d/2}$; on $F_2$ it is exponential, and after $400$ steps the return probability is below $10^{-28}$.

### P8.5

**How to find it.** Power iteration: $\pi\leftarrow0.85\,\pi P+0.15/N$, with pages without outgoing links treated as linking to everyone. Monte Carlo: from each page start $m$ walks; each walk follows links with probability $0.85$ per step and stops at the first teleportation; the visit counts, normalised, estimate $\pi$, because $\pi=\frac{0.15}N\sum_t0.85^t\mathbf 1^{\mathsf T}P^t$.

In [6]:
N, alpha = 2000, 0.85
outdeg = rng.zipf(2.2, N).clip(max=200)                 # heavy-tailed out-degrees
pop = rng.zipf(1.8, N).astype(float); pop /= pop.sum()  # some pages are more linked to
links = [rng.choice(N, size=d, replace=True, p=pop) for d in outdeg]
rows = np.concatenate([np.full(len(l), i) for i, l in enumerate(links)]); cols = np.concatenate(links)
P = sp.csr_matrix((np.concatenate([np.full(len(l), 1 / len(l)) for l in links]), (rows, cols)), shape=(N, N))
pr = np.full(N, 1 / N)
for it in range(100):
    new = alpha * (P.T @ pr) + (1 - alpha) / N
    if np.abs(new - pr).sum() < 1e-12: break
    pr = new
print(f"power iteration converged in {it} iterations; top 5 pages {np.argsort(-pr)[:5]}")

m = 20; visits = np.zeros(N)
for s in range(N):
    for _ in range(m):
        x = s; visits[x] += 1
        while rng.random() < alpha:
            l = links[x]; x = l[rng.integers(len(l))]; visits[x] += 1
mc = visits / visits.sum()
top = np.argsort(-pr)[:10]
print("PageRank of the top 10, power iteration:", np.round(pr[top], 4))
print("                         Monte Carlo   :", np.round(mc[top], 4))
print(f"total variation distance {0.5 * np.abs(mc - pr).sum():.4f}")

power iteration converged in 86 iterations; top 5 pages [1912 1986 1026 1989 1761]


PageRank of the top 10, power iteration: [0.0856 0.0847 0.0722 0.0568 0.0318 0.0282 0.0243 0.018  0.0158 0.0154]
                         Monte Carlo   : [0.0859 0.0851 0.0724 0.0569 0.0318 0.0284 0.0245 0.0179 0.0157 0.0153]
total variation distance 0.0073


The two methods agree on the important pages, which is what a search engine needs; the total variation distance is dominated by the many unimportant pages, each estimated from few visits. Power iteration converges geometrically at rate $0.85$ (the teleportation makes the chain uniformly ergodic), which is why Brin and Page chose a damping factor well below $1$ (S. Brin and L. Page, *The anatomy of a large-scale hypertextual Web search engine*, Computer Networks 30, 1998). The Monte Carlo version can be updated incrementally and run in parallel (K. Avrachenkov, N. Litvak, D. Nemirovsky and N. Osipova, *Monte Carlo methods in PageRank computation*, SIAM Journal on Numerical Analysis 45, 2007).

### P8.6

**How to find it.** $R_e=(e_a-e_b)^{\mathsf T}L^+(e_a-e_b)$ for $e=ab$, and $\sum_eR_e=n-1$ (Foster's theorem), so $p_e=R_e/(n-1)$. The quality of $\tilde L$ is measured by the spectrum of $L^{+/2}\tilde LL^{+/2}$ on the orthogonal of the constants: all its eigenvalues lie in $[1-\varepsilon,1+\varepsilon]$ exactly when $(1-\varepsilon)x^{\mathsf T}Lx\le x^{\mathsf T}\tilde Lx\le(1+\varepsilon)x^{\mathsf T}Lx$ for all $x$.

In [7]:
n = 300
h = n // 2
edges = [(i, j) for i in range(n) for j in range(i + 1, n)
         if (i < h) == (j < h) and rng.random() < 0.5]   # two dense communities
edges += [(0, h)]                                        # one bridge
edges = list({tuple(sorted(e)) for e in edges})
E = np.array(edges); m_e = len(E)
L = laplacian(n, edges); Lp = np.linalg.pinv(L)
Reff = Lp[E[:, 0], E[:, 0]] + Lp[E[:, 1], E[:, 1]] - 2 * Lp[E[:, 0], E[:, 1]]
print(f"{m_e} edges; sum of effective resistances {Reff.sum():.3f} (n - 1 = {n - 1})")

w, V = np.linalg.eigh(L); w[0] = np.inf
S = V @ np.diag(1 / np.sqrt(w)) @ V.T                     # L^{+1/2}
def quality(probs, q):
    idx = rng.choice(m_e, size=q, p=probs)
    Lt = laplacian(n, [tuple(E[i]) for i in idx], 1 / (q * probs[idx]))
    ev = np.linalg.eigvalsh(S @ Lt @ S)[1:]
    return max(ev.min(), 0.0), ev.max()

for q in (2000, 8000, 32000):
    lo_r, hi_r = quality(Reff / Reff.sum(), q)
    lo_u, hi_u = quality(np.full(m_e, 1 / m_e), q)
    print(f"q = {q:>5} samples: by resistance [{lo_r:.2f}, {hi_r:.2f}]   uniform [{lo_u:.2f}, {hi_u:.2f}]")

11335 edges; sum of effective resistances 299.000 (n - 1 = 299)


q =  2000 samples: by resistance [0.29, 2.21]   uniform [0.00, 2.09]


q =  8000 samples: by resistance [0.61, 1.49]   uniform [0.00, 1.47]


q = 32000 samples: by resistance [0.80, 1.23]   uniform [0.00, 1.26]


Sampling by effective resistance gives a spectral approximation whose error shrinks like $\sqrt{(n\log n)/q}$, as Spielman and Srivastava proved (*Graph sparsification by effective resistances*, SIAM Journal on Computing 40, 2011; STOC 2008), with a number of samples independent of the number of edges. Uniform sampling fails on this graph. The bridge is one edge among $11\,335$, so $q$ uniform samples hit it about $q/11\,335$ times, fewer than $3$ times even for $q=32\,000$, and here it was missed every time: the sparsified graph falls apart into two pieces, the cut between the communities gets weight $0$, and the lower end of the spectrum is $0$. Sampling by resistance sees that the bridge is indispensable: its resistance is $R_e=1$, the largest possible, so it is drawn with probability $\frac1{299}$ per sample and is kept every time, with the right weight on average. The effective resistance $R_e$ of an edge is the probability that it belongs to a uniform random spanning tree (Kirchhoff), the measure of how indispensable it is. Sparsifiers are a key step in the nearly linear time Laplacian solvers of Spielman and Teng.